

This notebook basically does parts/steps 3-5 from our given project requirements. It does not (yet) include to the unsupervised learning you have already, also it does no include any sensitivity checks that might be a good idea after all. 
 

#### Predictive question

> Can we predict whether a hotel booking will be canceled using customer and reservation information plausibly available at booking time?

The response is:

$$
Y_i = \texttt{is\_canceled}_i \in \{0,1\}
$$

where `1` means canceled and `0` means not canceled.

The logistic regression:

$$
P(Y_i=1 \mid x_i)=\pi_i = \frac{\exp(x_i^T\beta)}{1+\exp(x_i^T\beta)},
\qquad
\log\left(\frac{\pi_i}{1-\pi_i}\right)=x_i^T\beta.
$$

The L2-penalty shrinks coefficients through, like we know from basic ridge regression:

$$
-\ell(\beta)+\lambda\sum_{j=1}^p\beta_j^2.
$$


In [1]:
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


def get_scores(estimator, X_eval):
    if hasattr(estimator, 'predict_proba'):
        return estimator.predict_proba(X_eval)[:, 1]
    if hasattr(estimator, 'decision_function'):
        scores = estimator.decision_function(X_eval)
        return 1 / (1 + np.exp(-scores))
    return None


def evaluate_classifier(name, estimator, X_eval, y_eval, threshold=0.5):
    y_score = get_scores(estimator, X_eval)
    if y_score is not None:
        y_pred = (y_score >= threshold).astype(int)
        auc = roc_auc_score(y_eval, y_score)
    else:
        y_pred = estimator.predict(X_eval)
        auc = np.nan
    cm = confusion_matrix(y_eval, y_pred, labels=[0, 1])
    metrics = {
        'model': name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_eval, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_eval, y_pred),
        'precision': precision_score(y_eval, y_pred, zero_division=0),
        'recall_sensitivity': recall_score(y_eval, y_pred, zero_division=0),
        'specificity': specificity_score(y_eval, y_pred),
        'f1': f1_score(y_eval, y_pred, zero_division=0),
        'roc_auc': auc,
    }
    return metrics, cm, y_score, y_pred

In [2]:
from pathlib import Path
import json
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

### Feature  preprocessing

This section creates the design matrix. PLEASE ADJUST DATA PATH FOR RE-RUNNING (I forgot to change my working directory at the beginning)

In [3]:
df = pd.read_csv(r"C:\Users\Lenovo\Documents\Columbia\Spring 26\machine learning\project\hotel-booking-project-main\data\processed\hotel_bookings_cleaned.csv")
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
display(df.head())

Shape: 118,563 rows x 40 columns


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_nights,total_guests,is_family,has_agent,has_company,room_changed,is_duplicate_record
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0,0,0,Transient,75.0000,0,0,Check-Out,2015-07-02,2015-07-01,1,1,0,0,0,1,0
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,0,0,Transient,75.0000,0,0,Check-Out,2015-07-02,2015-07-01,1,1,0,1,0,0,0
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,0,0,Transient,98.0000,0,1,Check-Out,2015-07-03,2015-07-01,2,2,0,1,0,0,1
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,0,0,Transient,98.0000,0,1,Check-Out,2015-07-03,2015-07-01,2,2,0,1,0,0,1
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,0,No Deposit,0,0,0,Transient,107.0000,0,0,Check-Out,2015-07-03,2015-07-01,2,2,0,0,0,0,0


In [4]:
target_col = 'is_canceled'
if target_col not in df.columns:
    raise ValueError(f'Missing target column: {target_col}')
if set(df[target_col].dropna().unique()) - {0, 1}:
    raise ValueError('Target column is not binary 0/1.')

target_summary = pd.DataFrame({
    'count': df[target_col].value_counts(dropna=False).sort_index(),
    'proportion': df[target_col].value_counts(normalize=True, dropna=False).sort_index(),
})
display(target_summary)

,count,proportion
is_canceled,,
0,74388,0.6274
1,44175,0.3726




The main model uses a "at-booking feature" set.

Excluded columns:

| Column/group | Reason |
|---|---|
| `reservation_status`, `reservation_status_date` | direct outcome leakage |
| `assigned_room_type`, `room_changed` | likely post-booking information |
| `booking_changes` | changes can happen after initial booking |
| `deposit_type` | suspicious in the EDA; used only in a sensitivity check |
| `is_duplicate_record` | cleaning/audit flag |
| raw `arrival_date` | replaced by compact date-derived features |

In [5]:
df_model = df.copy()

if 'arrival_date' in df_model.columns:
    arrival = pd.to_datetime(df_model['arrival_date'], errors='coerce')
    df_model['arrival_month_num'] = arrival.dt.month
    df_model['arrival_quarter'] = arrival.dt.quarter
    df_model['arrival_dayofweek'] = arrival.dt.dayofweek
    df_model['arrival_is_weekend'] = arrival.dt.dayofweek.isin([5, 6]).astype(int)

strict_leakage_cols = ['reservation_status', 'reservation_status_date']
post_booking_cols = ['assigned_room_type', 'room_changed', 'booking_changes']
problematic_or_audit_cols = ['deposit_type', 'is_duplicate_record']
raw_date_cols = ['arrival_date']
eda_helper_cols = [c for c in df_model.columns if c.endswith('_bin') or c in ['waiting_list_bin']]

exclude_cols = list(dict.fromkeys([
    target_col,
    *strict_leakage_cols,
    *post_booking_cols,
    *problematic_or_audit_cols,
    *raw_date_cols,
    *eda_helper_cols,
]))
exclude_cols = [c for c in exclude_cols if c in df_model.columns]
features = [c for c in df_model.columns if c not in exclude_cols]

print('Excluded columns:')
display(exclude_cols)
print(f'Number of features before one-hot encoding: {len(features)}')
display(features)

X = df_model[features].copy()
y = df_model[target_col].astype(int).copy()

Excluded columns:


['is_canceled',
 'reservation_status',
 'reservation_status_date',
 'assigned_room_type',
 'room_changed',
 'booking_changes',
 'deposit_type',
 'is_duplicate_record',
 'arrival_date']

Number of features before one-hot encoding: 35


['hotel',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_nights',
 'total_guests',
 'is_family',
 'has_agent',
 'has_company',
 'arrival_month_num',
 'arrival_quarter',
 'arrival_dayofweek',
 'arrival_is_weekend']

### Seasonality-balanced train/test split


Here the test set is  by the combination of cancellation status and arrival month (both training and test set matching the overall average responses of the variables).This is because of our concern that a random split might accidentally put too much high-season or low-season data into one side of the split. The seasonality is determined by arrival month (`arrival_month_num`). It is an 80-20 split.

In [6]:

if 'arrival_month_num' not in df_model.columns:
    raise ValueError('arrival_month_num is required for the seasonality-balanced split.')

split_meta = df_model.loc[X.index, [target_col, 'arrival_month_num']].copy()
split_meta['arrival_month_num'] = split_meta['arrival_month_num'].astype('Int64')

# Combined stratification label: target + month.
stratify_label = (
    split_meta[target_col].astype(str)
    + '_month_'
    + split_meta['arrival_month_num'].astype(str)
)

# Safety check-- train_test_split with stratify requires at least 2 observations per stratum.

stratum_counts = stratify_label.value_counts()
if stratum_counts.min() < 2:
    print('Some target-month strata are too small; falling back to target-quarter stratification.')
    if 'arrival_quarter' not in df_model.columns:
        raise ValueError('arrival_quarter is required for the fallback split.')
    stratify_label = (
        split_meta[target_col].astype(str)
        + '_quarter_'
        + df_model.loc[X.index, 'arrival_quarter'].astype('Int64').astype(str)
    )

train_idx, test_idx = train_test_split(
    X.index,
    test_size=0.20,
    random_state=42,
    stratify=stratify_label,
)

X_train = X.loc[train_idx].copy()
X_test = X.loc[test_idx].copy()
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

#  Cancellation-rate balance.
target_balance = pd.DataFrame({
    'train': y_train.value_counts(normalize=True).sort_index(),
    'test': y_test.value_counts(normalize=True).sort_index(),
}).fillna(0)
target_balance['absolute_difference'] = (target_balance['train'] - target_balance['test']).abs()
print('Cancellation-rate balance:')
display(target_balance)

#  Arrival-month balance.
month_balance = pd.DataFrame({
    'train': df_model.loc[train_idx, 'arrival_month_num'].value_counts(normalize=True).sort_index(),
    'test': df_model.loc[test_idx, 'arrival_month_num'].value_counts(normalize=True).sort_index(),
}).fillna(0)
month_balance['absolute_difference'] = (month_balance['train'] - month_balance['test']).abs()
print('Arrival-month balance:')
display(month_balance)

print(
    'Maximum absolute train/test month-share difference:',
    f"{month_balance['absolute_difference'].max():.4f}"
)


# This checks whether the positive-class prevalence within months is also reasonably comparable.
monthly_cancellation_balance = pd.DataFrame({
    'train_cancel_rate': df_model.loc[train_idx].groupby('arrival_month_num')[target_col].mean(),
    'test_cancel_rate': df_model.loc[test_idx].groupby('arrival_month_num')[target_col].mean(),
})
monthly_cancellation_balance['absolute_difference'] = (
    monthly_cancellation_balance['train_cancel_rate']
    - monthly_cancellation_balance['test_cancel_rate']
).abs()
print('Cancellation-rate-by-month balance:')
display(monthly_cancellation_balance)


Training shape: (94850, 35)
Test shape: (23713, 35)
Cancellation-rate balance:


,train,test,absolute_difference
is_canceled,,,
0,0.6274,0.6275,0.0001
1,0.3726,0.3725,0.0001


Arrival-month balance:


,train,test,absolute_difference
arrival_month_num,,,
1,0.0495,0.0496,0.0000
2,0.0675,0.0675,0.0000
3,0.0819,0.0819,0.0000
4,0.0932,0.0932,0.0000
5,0.0987,0.0987,0.0000
6,0.0919,0.0918,0.0000
7,0.1062,0.1061,0.0000
8,0.1165,0.1165,0.0000
9,0.0884,0.0884,0.0000


Maximum absolute train/test month-share difference: 0.0000
Cancellation-rate-by-month balance:


,train_cancel_rate,test_cancel_rate,absolute_difference
arrival_month_num,,,
1,0.3071,0.3072,0.0002
2,0.3359,0.3356,0.0003
3,0.3238,0.3237,0.0000
4,0.4088,0.4088,0.0000
5,0.3994,0.3994,0.0000
6,0.4162,0.4160,0.0002
7,0.3763,0.3762,0.0001
8,0.3792,0.3791,0.0001
9,0.3927,0.3927,0.0000


### Preprocessing 

The EDA showed that the cleaned dataset has no actual missing cells. Therefore we do not have to perform any kind of imputation.
What we do is the following: 

We determine that `agent` and `company` are categorical variables despite their numerical values. We standardize numerical variable (as is advisable for L2-regularized logistic regression).
We numerically encode the remaining categorical variables, because for the logistic regression we need a numerical matrix.

The preprocessing is apllied to the training data only at first.

In [7]:
# Validate the EDA conclusion: no actual NaNs in the modeling matrices.
missing_train = int(X_train.isna().sum().sum())
missing_test = int(X_test.isna().sum().sum())
print('Number of missing cells in X_train:', missing_train)
print('Number of missing cells in X_test:', missing_test)

if missing_train or missing_test:
    missing_counts = X_train.isna().sum().sort_values(ascending=False)
    display(missing_counts[missing_counts > 0].to_frame('missing_count'))
    raise ValueError('Unexpected missing values found. Inspect them instead of silently imputing.')

# Numeric-looking ID columns are categorical labels. Keep 0 as an explicit category.
for frame in [X_train, X_test]:
    for col in [c for c in ['agent', 'company'] if c in frame.columns]:
        frame[col] = frame[col].astype('Int64').astype(str)

numeric_features = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print('Numeric features:')
display(numeric_features)
print('Categorical features:')
display(categorical_features)

numeric_transformer = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
])

try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=True, min_frequency=10)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=True)

categorical_transformer = Pipeline([
    ('onehot', onehot),
])

preprocess = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop', sparse_threshold=1.0)

X_train_pp = preprocess.fit_transform(X_train)
X_test_pp = preprocess.transform(X_test)

# Force efficient row slicing for cross-validation on sparse matrices.
if hasattr(X_train_pp, 'tocsr'):
    X_train_pp = X_train_pp.tocsr()
if hasattr(X_test_pp, 'tocsr'):
    X_test_pp = X_test_pp.tocsr()

print('Training shape before preprocessing:', X_train.shape)
print('Test shape before preprocessing:', X_test.shape)
print('Training shape after preprocessing:', X_train_pp.shape)
print('Test shape after preprocessing:', X_test_pp.shape)

Number of missing cells in X_train: 0
Number of missing cells in X_test: 0
Numeric features:


['lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_nights',
 'total_guests',
 'is_family',
 'has_agent',
 'has_company',
 'arrival_month_num',
 'arrival_quarter',
 'arrival_dayofweek',
 'arrival_is_weekend']

Categorical features:


['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'agent',
 'company',
 'customer_type']

Training shape before preprocessing: (94850, 35)
Test shape before preprocessing: (23713, 35)
Training shape after preprocessing: (94850, 478)
Test shape after preprocessing: (23713, 478)


### Model development

The main model is L2-regularized logistic regression. A majority baseline and a simple numeric-only decision tree are included forcomparision.

In [8]:
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


def get_scores(estimator, X_eval):
    if hasattr(estimator, 'predict_proba'):
        return estimator.predict_proba(X_eval)[:, 1]
    if hasattr(estimator, 'decision_function'):
        scores = estimator.decision_function(X_eval)
        return 1 / (1 + np.exp(-scores))
    return None


def evaluate_classifier(name, estimator, X_eval, y_eval, threshold=0.5):
    y_score = get_scores(estimator, X_eval)
    if y_score is not None:
        y_pred = (y_score >= threshold).astype(int)
        auc = roc_auc_score(y_eval, y_score)
    else:
        y_pred = estimator.predict(X_eval)
        auc = np.nan
    cm = confusion_matrix(y_eval, y_pred, labels=[0, 1])
    metrics = {
        'model': name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_eval, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_eval, y_pred),
        'precision': precision_score(y_eval, y_pred, zero_division=0),
        'recall_sensitivity': recall_score(y_eval, y_pred, zero_division=0),
        'specificity': specificity_score(y_eval, y_pred),
        'f1': f1_score(y_eval, y_pred, zero_division=0),
        'roc_auc': auc,
    }
    return metrics, cm, y_score, y_pred

### Basline model

This is basically a "dumb" comparision model that just predicts "is canceled" every time, whose accuary would thus be the response mean of $67.5 \% $. So we know our actual model must be at least as good or we are not on the right track.

In [9]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_pp, y_train)
baseline_metrics, baseline_cm, baseline_score, baseline_pred = evaluate_classifier(
    'majority_baseline', baseline, X_test_pp, y_test
)
display(pd.DataFrame([baseline_metrics]))
print('Baseline confusion matrix:')
display(pd.DataFrame(
    baseline_cm,
    index=['actual_not_canceled', 'actual_canceled'],
    columns=['pred_not_canceled', 'pred_canceled'],
))

,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,majority_baseline,0.5000,0.6275,0.5000,0.0000,0.0000,1.0000,0.0000,0.5000


Baseline confusion matrix:


,pred_not_canceled,pred_canceled
actual_not_canceled,14879,0
actual_canceled,8834,0


### Numerical values decision tree

This is a simple nonlinear model for comparision that is easy to get via scikitlearn.

In [10]:
# The numeric columns are first in X_train_pp because they are the first transformer in ColumnTransformer.
n_numeric = len(numeric_features)
X_train_num_pp = X_train_pp[:, :n_numeric]
X_test_num_pp = X_test_pp[:, :n_numeric]

numeric_tree_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=250,
    random_state=42,
)
numeric_tree_model.fit(X_train_num_pp, y_train)
print('Numeric-only decision-tree comparator fitted with max_depth=8 and min_samples_leaf=250.')

Numeric-only decision-tree comparator fitted with max_depth=8 and min_samples_leaf=250.


We can directly run it on the test set and output the scores:

In [11]:
tree_metrics, tree_cm, tree_score, tree_pred = evaluate_classifier(
    'numeric_decision_tree_comparator',
    numeric_tree_model,
    X_test_num_pp,
    y_test,
    threshold=0.5,
)

display(pd.DataFrame([tree_metrics]))

print('Decision-tree comparator test performance')
print(f"Accuracy:           {tree_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:  {tree_metrics['balanced_accuracy']:.4f}")
print(f"Recall/sensitivity: {tree_metrics['recall_sensitivity']:.4f}")
print(f"Specificity:        {tree_metrics['specificity']:.4f}")
print(f"F1:                 {tree_metrics['f1']:.4f}")
print(f"ROC-AUC:            {tree_metrics['roc_auc']:.4f}")

,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,numeric_decision_tree_comparator,0.5000,0.7553,0.7138,0.7259,0.5513,0.8764,0.6266,0.8172


Decision-tree comparator test performance
Accuracy:           0.7553
Balanced accuracy:  0.7138
Recall/sensitivity: 0.5513
Specificity:        0.8764
F1:                 0.6266
ROC-AUC:            0.8172


### L2-regularized logistic regression

See formula in cell 1 above.

In scikit-learn, `C` is the inverse regularization strength: smaller `C` means stronger regularization.

The preprocessing map is learned once from the training set, and cross-validation tunes `C` on the preprocessed training matrix. 

In [19]:
from sklearn.model_selection import cross_val_score

cv_stratify_label = (
    df_model.loc[X_train.index, target_col].astype(str)
    + '_month_'
    + df_model.loc[X_train.index, 'arrival_month_num'].astype('Int64').astype(str)
)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_splits = list(inner_cv.split(X_train_pp, cv_stratify_label))

candidate_C = [0.01, 0.1]
cv_rows = []

for C in candidate_C:
    candidate_model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=100,
        random_state=42,
    )
    scores = cross_val_score(
        candidate_model,
        X_train_pp,
        y_train,
        cv=cv_splits,
        scoring='roc_auc',
        n_jobs=1,
    )
    cv_rows.append({
        'C': C,
        'mean_test_roc_auc': scores.mean(),
        'std_test_roc_auc': scores.std(),
    })

l2_cv_results = pd.DataFrame(cv_rows).sort_values('mean_test_roc_auc', ascending=False).reset_index(drop=True)
l2_cv_results['rank_test_score'] = np.arange(1, len(l2_cv_results) + 1)
display(l2_cv_results)

best_C = float(l2_cv_results.iloc[0]['C'])
print('Selected C:', best_C)

l2_model = LogisticRegression(
    solver='liblinear',
    C=best_C,
    max_iter=100,
    random_state=42,
)
l2_model.fit(X_train_pp, y_train)
print('L2 logistic-regression model fitted.')

,C,mean_test_roc_auc,std_test_roc_auc,rank_test_score
0,0.1000,0.8926,0.0021,1
1,0.0100,0.8886,0.0034,2


Selected C: 0.1
L2 logistic-regression model fitted.


### L2 logistic model performance

This cell prints the selected L2 logistic regression model's held-out test performance immediately after the model is fitted.

In [13]:
l2_metrics, l2_cm, l2_score, l2_pred = evaluate_classifier(
    'logistic_l2_main',
    l2_model,
    X_test_pp,
    y_test,
    threshold=0.5,
)

display(pd.DataFrame([l2_metrics]))

print('Main L2 logistic regression test performance')
print(f"Accuracy:           {l2_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:  {l2_metrics['balanced_accuracy']:.4f}")
print(f"Recall/sensitivity: {l2_metrics['recall_sensitivity']:.4f}")
print(f"Specificity:        {l2_metrics['specificity']:.4f}")
print(f"F1:                 {l2_metrics['f1']:.4f}")
print(f"ROC-AUC:            {l2_metrics['roc_auc']:.4f}")

,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,logistic_l2_main,0.5000,0.8108,0.7879,0.7723,0.6980,0.8778,0.7333,0.8910


Main L2 logistic regression test performance
Accuracy:           0.8108
Balanced accuracy:  0.7879
Recall/sensitivity: 0.6980
Specificity:        0.8778
F1:                 0.7333
ROC-AUC:            0.8910


### Model comparision

In [14]:
models = {
    'majority_baseline': (baseline, X_test_pp),
    'logistic_l2_main': (l2_model, X_test_pp),
    'numeric_decision_tree_comparator': (numeric_tree_model, X_test_num_pp),
}

rows = []
model_outputs = {}
for name, (model, X_eval) in models.items():
    metrics, cm, y_score, y_pred = evaluate_classifier(name, model, X_eval, y_test)
    rows.append(metrics)
    model_outputs[name] = {'cm': cm, 'y_score': y_score, 'y_pred': y_pred}

metrics_df = pd.DataFrame(rows).sort_values('roc_auc', ascending=False).reset_index(drop=True)
display(metrics_df)

for name in ['logistic_l2_main', 'numeric_decision_tree_comparator']:
    print()
    print('=' * 80)
    print(name)
    cm_df = pd.DataFrame(
        model_outputs[name]['cm'],
        index=['actual_not_canceled', 'actual_canceled'],
        columns=['pred_not_canceled', 'pred_canceled'],
    )
    display(cm_df)

,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,logistic_l2_main,0.5000,0.8108,0.7879,0.7723,0.6980,0.8778,0.7333,0.8910
1,numeric_decision_tree_comparator,0.5000,0.7553,0.7138,0.7259,0.5513,0.8764,0.6266,0.8172
2,majority_baseline,0.5000,0.6275,0.5000,0.0000,0.0000,1.0000,0.0000,0.5000



logistic_l2_main


,pred_not_canceled,pred_canceled
actual_not_canceled,13061,1818
actual_canceled,2668,6166



numeric_decision_tree_comparator


,pred_not_canceled,pred_canceled
actual_not_canceled,13040,1839
actual_canceled,3964,4870


In [20]:
display(
    metrics_df[[
        'model',
        'accuracy',
        'balanced_accuracy',
        'precision',
        'recall_sensitivity',
        'specificity',
        'f1',
        'roc_auc',
    ]].sort_values('roc_auc', ascending=False)
)

,model,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,logistic_l2_main,0.8108,0.7879,0.7723,0.6980,0.8778,0.7333,0.8910
1,numeric_decision_tree_comparator,0.7553,0.7138,0.7259,0.5513,0.8764,0.6266,0.8172
2,majority_baseline,0.6275,0.5000,0.0000,0.0000,1.0000,0.0000,0.5000


In [21]:
selected_model_name = 'logistic_l2_main'
selected_model = l2_model
selected_metrics = metrics_df.loc[metrics_df['model'] == selected_model_name]

print('Selected final model:', selected_model_name)
print('Selected C:', best_C)
display(selected_metrics)
print('Selected L2 confusion matrix:')
display(pd.DataFrame(
    model_outputs[selected_model_name]['cm'],
    index=['actual_not_canceled', 'actual_canceled'],
    columns=['pred_not_canceled', 'pred_canceled'],
))

Selected final model: logistic_l2_main
Selected C: 0.1


,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1,roc_auc
0,logistic_l2_main,0.5000,0.8108,0.7879,0.7723,0.6980,0.8778,0.7333,0.8910


Selected L2 confusion matrix:


,pred_not_canceled,pred_canceled
actual_not_canceled,13061,1818
actual_canceled,2668,6166


###  Interpret final logistic model

The coefficients are log-odds coefficients. Their exponentiated values are odds ratios.

In [17]:
def get_feature_names_from_preprocessor(preprocessor):
    names = []
    for transformer_name, transformer, columns in preprocessor.transformers_:
        if transformer_name == 'remainder' or transformer == 'drop':
            continue
        if transformer_name == 'num':
            names.extend(list(columns))
        elif transformer_name == 'cat':
            ohe = transformer.named_steps['onehot']
            try:
                names.extend(list(ohe.get_feature_names_out(columns)))
            except AttributeError:
                names.extend(list(ohe.get_feature_names(columns)))
    return np.array(names)

feature_names = get_feature_names_from_preprocessor(preprocess)
coefs = selected_model.coef_.ravel()

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coef_log_odds': coefs,
    'odds_ratio': np.exp(coefs),
    'abs_coef': np.abs(coefs),
}).sort_values('abs_coef', ascending=False).reset_index(drop=True)

print(f'Number of post-encoding features: {len(coef_table):,}')
display(coef_table.head(30))

Number of post-encoding features: 478


,feature,coef_log_odds,odds_ratio,abs_coef
0,country_PRT,1.8497,6.3580,1.8497
1,previous_cancellations,1.7067,5.5109,1.7067
2,required_car_parking_spaces,-1.4022,0.2461,1.4022
3,agent_240,1.3380,3.8115,1.3380
4,agent_9,1.1872,3.2780,1.1872
5,customer_type_Transient,0.9534,2.5945,0.9534
6,agent_7,-0.8812,0.4143,0.8812
7,country_DEU,-0.8026,0.4482,0.8026
8,market_segment_Groups,0.7449,2.1063,0.7449
9,lead_time,0.7027,2.0192,0.7027


In [18]:
# Save outputs to the current working directory.

output_dir = Path.cwd()

metrics_df.to_csv("test_set_model_comparison.csv", index=False)
coef_table.to_csv("selected_l2_logistic_coefficients.csv", index=False)
l2_cv_results.to_csv("l2_logistic_cv_results.csv", index=False)
target_balance.to_csv("split_target_balance.csv")
month_balance.to_csv("split_arrival_month_balance.csv")
monthly_cancellation_balance.to_csv("split_monthly_cancellation_rate_balance.csv")

selected_pred_df = pd.DataFrame({
    "y_true": y_test.to_numpy(),
    "y_pred": model_outputs[selected_model_name]["y_pred"],
    "predicted_cancellation_probability": model_outputs[selected_model_name]["y_score"],
})
selected_pred_df.to_csv("selected_l2_test_predictions.csv", index=False)

summary = {
    "data_shape": df.shape,
    "total_nan_cells_in_cleaned_file": int(df.isna().sum().sum()),
    "excluded_columns": exclude_cols,
    "features_before_encoding": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "train_shape_before_preprocessing": X_train.shape,
    "test_shape_before_preprocessing": X_test.shape,
    "train_shape_after_preprocessing": X_train_pp.shape,
    "test_shape_after_preprocessing": X_test_pp.shape,
    "selected_model": selected_model_name,
    "selected_C": best_C,
}

with open("modeling_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

zip_path = Path("modeling_outputs_steps_3_to_5_season_balanced_no_imputation.zip")

files_to_zip = [
    "test_set_model_comparison.csv",
    "selected_l2_logistic_coefficients.csv",
    "l2_logistic_cv_results.csv",
    "split_target_balance.csv",
    "split_arrival_month_balance.csv",
    "split_monthly_cancellation_rate_balance.csv",
    "selected_l2_test_predictions.csv",
    "modeling_summary.json",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for filename in files_to_zip:
        path = Path(filename)
        if path.exists():
            z.write(path, arcname=path.name)

print("Saved outputs in current working directory:")
print(output_dir)

for filename in files_to_zip:
    print("-", filename)

print("ZIP:", zip_path.name)

Saved outputs in current working directory:
c:\Users\Lenovo\Downloads
- test_set_model_comparison.csv
- selected_l2_logistic_coefficients.csv
- l2_logistic_cv_results.csv
- split_target_balance.csv
- split_arrival_month_balance.csv
- split_monthly_cancellation_rate_balance.csv
- selected_l2_test_predictions.csv
- modeling_summary.json
ZIP: modeling_outputs_steps_3_to_5_season_balanced_no_imputation.zip


## Quick summary

The cleaned dataset had no actual missing values, so no substantive imputation was performed. Numerical predictors were standardized for L2-regularized logistic regression, categorical predictors were numerically encoded, and numeric-looking ID columns such as `agent` and `company` were treated as categorical variables. The train/test split was stratified jointly by cancellation status and arrival month to keep both the cancellation rate and seasonal composition balanced. The main model was an L2-regularized logistic regression, selected because it provides interpretable cancellation probabilities and odds-ratio interpretation while. It also helps controlling overfitting by means of regularization. We compared it with a majority baseline and a numeric-only decision tree. Hyperparameters were tuned with cross-validation on the training data, and final performance was assessed on a test set. 